Just a bootstrap to fix the relative path issue.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path("/home/xavier/dev/insurance-claims-multiagent-rag")
os.chdir(ROOT)
for p in (ROOT, ROOT / "app" / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("cwd:", Path.cwd())
print("python:", sys.version.split()[0])

Connectivity and migration state

In [ ]:
from sqlalchemy import inspect, text

from infrastructure.database import (
    ChunkRow,
    create_engine_from_settings,
    create_session_factory,
    is_database_reachable,
    upsert_chunks,
)

engine = create_engine_from_settings()
Session = create_session_factory(engine=engine)

print("reachable:", is_database_reachable(Session))

inspector = inspect(engine)
print("chunk table present:", inspector.has_table("chunk"))

with Session() as s:
    version = s.execute(text("SELECT version_num FROM alembic_version")).scalar_one()
print("alembic head:", version)


In [ ]:
cols = {c["name"]: c for c in inspector.get_columns("chunk")}
print("bundle_section nullable:", cols["bundle_section"]["nullable"],
      "| default:", cols["bundle_section"]["default"])
print("confidence nullable:", cols["confidence"]["nullable"])
print("has embedded_text + display_text:",
      "embedded_text" in cols and "display_text" in cols)

print("\nindexes:")
for ix in inspector.get_indexes("chunk"):
    print(" ", ix["name"], "-", ix["column_names"])

print("\nchecks:")
for ck in inspector.get_check_constraints("chunk"):
    print(" ", ck["name"])


In [ ]:
from domain.chunk import Chunk, ChunkRule
from domain.clause_classification import ClauseProvenance, ClauseType, TypeSource
from infrastructure.rag.chunk_schema import flatten_chunk

provenance = ClauseProvenance(
    document_id="1", susep_process="15414.900666/2014-89", insurer="Bradesco Seguros",
    cnpj="12345678000199", product_line="CASCO", indemnity_regime="VD", process_year="2019",
)

chunk = Chunk(
    document_id="1", chunk_id="1:9.2.2", clause_id="1:9.2.2",
    source_clause_ids=("1:9.2.2",), chunk_index=0, chunk_count=1,
    parent_path="9. COBERTURAS BÁSICAS > 9.2 R.C.F.V.",
    text="9. COBERTURAS BÁSICAS > 9.2 R.C.F.V.\n9.2.2 Riscos Cobertos\n\nDanos a terceiros...",
    char_count=79, rule=ChunkRule.SINGLE, clause_type=ClauseType.COVERAGE,
    type_source=TypeSource.RULE, confidence=1.0, bundle_section=None, provenance=provenance,
)

record = flatten_chunk(chunk, source="text")
print("source:", record.source)
print("embedded (text):", repr(record.text))
print("display_text:", repr(record.display_text))
print("invariant holds:", record.text == f"{record.parent_path}\n{record.display_text}")


In [ ]:
from scripts.build_chunks import resolve_source
from infrastructure.parsing.manifest import read_manifest

manifest = read_manifest(Path("data/policies/manifest.csv"))
for doc_id in ("1", "25"):
    entry = next(e for e in manifest if e["id"] == doc_id)
    print(f"doc {doc_id}: extraction_mode={entry['extraction_mode']!r}"
          f" - source={resolve_source(entry['extraction_mode'])!r}")


In [ ]:
from sqlalchemy import func, select


def make_record(**overrides):
    """A valid ChunkRecord with nbtest: ids; override any field."""
    base = dict(
        schema_version="v1", chunk_id="nbtest:1", document_id="1", clause_id="nbtest:1",
        source_clause_ids=["nbtest:1"], chunk_index=0, chunk_count=1,
        parent_path="1. CONDIÇÕES GERAIS",
        text="1. CONDIÇÕES GERAIS\n2. COBERTURAS\n\nTexto.",
        display_text="2. COBERTURAS\n\nTexto.", char_count=42,
        rule="single", clause_type="coverage", type_source="rule",
        confidence=None, bundle_section=None, source="text",
        susep_process="15414.900666/2014-89", insurer="Bradesco Seguros",
        cnpj="12345678000199", product_line="CASCO", indemnity_regime="VD",
        filing_year="2019",
    )
    base.update(overrides)
    from infrastructure.rag.chunk_schema import ChunkRecord
    return ChunkRecord.model_validate(base)


def count() -> int:
    with Session() as s:
        return s.execute(
            select(func.count()).select_from(ChunkRow).where(ChunkRow.chunk_id.like("nbtest:%"))
        ).scalar_one()


records = [make_record(chunk_id="nbtest:1"), make_record(chunk_id="nbtest:2", clause_id="nbtest:2")]

with Session() as s:
    upsert_chunks(s, records)
    s.commit()
    upsert_chunks(s, records)
    s.commit()

print("rows after two identical writes:", count(), "(expect 2)")


In [ ]:
with Session() as s:
    upsert_chunks(s, [make_record(chunk_id="nbtest:1", display_text="Texto corrigido.", confidence=0.5)])
    s.commit()

with Session() as s:
    row = s.execute(select(ChunkRow).where(ChunkRow.chunk_id == "nbtest:1")).scalar_one()
    print("row count still:", count(), "(expect 2)")
    print("display_text now:", repr(row.display_text))
    print("confidence now:", row.confidence)


In [ ]:
with Session() as s:
    upsert_chunks(s, [
        make_record(chunk_id="nbtest:b1", bundle_section="Motocicletas"),
        make_record(chunk_id="nbtest:b2", bundle_section="Motocicletas"),
        make_record(chunk_id="nbtest:b3", bundle_section=None),
    ])
    s.commit()

with Session() as s:
    strict = s.execute(text(
        "SELECT chunk_id FROM chunk WHERE chunk_id LIKE 'nbtest:b%' AND bundle_section = :x"
    ), {"x": "Motocicletas"}).scalars().all()
    with_fallback = s.execute(text(
        "SELECT chunk_id FROM chunk WHERE chunk_id LIKE 'nbtest:b%' "
        "AND (bundle_section = :x OR bundle_section IS NULL)"
    ), {"x": "Motocicletas"}).scalars().all()

print("strict  = :x -", sorted(strict))
print("... OR IS NULL -", sorted(with_fallback))


In [ ]:
from sqlalchemy.exc import IntegrityError

bad = make_record(chunk_id="nbtest:bad").model_dump(mode="json")
bad["embedded_text"] = bad.pop("text")
bad["rule"] = "not_a_rule"

try:
    with Session() as s:
        s.execute(ChunkRow.__table__.insert(), [bad])
        s.commit()
    print("BUG: bad rule was accepted")
except IntegrityError as exc:
    print("rejected as expected:", str(exc.orig).splitlines()[0])


In [ ]:
from sqlalchemy.exc import IntegrityError

bad = make_record(chunk_id="nbtest:bad").model_dump(mode="json")
bad["embedded_text"] = bad.pop("text")
bad["rule"] = "not_a_rule"

try:
    with Session() as s:
        s.execute(ChunkRow.__table__.insert(), [bad])
        s.commit()
    print("BUG: bad rule was accepted")
except IntegrityError as exc:
    print("rejected as expected:", str(exc.orig).splitlines()[0])


Real chunks through the pipeline (no LLM)

In [ ]:
from application.use_cases.chunking import chunk_typed_clauses
from application.use_cases.clause_classification import classify_and_enrich_clauses
from application.use_cases.clause_segmentation import CLAUSE_SEGMENTATION_VERSION
from infrastructure.config.settings import get_chunking_settings
from infrastructure.parsing.clause_tree_caching import (
    clause_tree_cache_path,
    compute_clause_tree_cache_key,
    read_clause_tree_cache,
)
from infrastructure.parsing.rules_loader import load_classification_rules

rules = load_classification_rules(Path("data/parsing/clause_type_mapping.csv"))
cs = get_chunking_settings()
cache_key = compute_clause_tree_cache_key(CLAUSE_SEGMENTATION_VERSION)


class RuleOnly:
    def classify(self, title, body):
        return ClauseType.OTHER, 0.0


all_records = []
for doc_id in ("1", "25"):
    entry = next(e for e in manifest if e["id"] == doc_id)
    src = resolve_source(entry["extraction_mode"])
    tree = read_clause_tree_cache(clause_tree_cache_path(doc_id, cache_key))
    typed = classify_and_enrich_clauses(tree, manifest, rules, RuleOnly())
    chunks, _ = chunk_typed_clauses(
        typed,
        min_char_count=cs.chunk_min_char_count,
        target_char_count=cs.chunk_target_char_count,
        max_char_count=cs.chunk_max_char_count,
        sliding_window_overlap_chars=cs.chunk_sliding_window_overlap_chars,
    )
    recs = [flatten_chunk(c, source=src) for c in chunks]
    recs = [r.model_copy(update={"chunk_id": f"nbtest:real:{r.chunk_id}"}) for r in recs]
    all_records.extend(recs)
    print(f"doc {doc_id}: source={src!r}, {len(recs)} chunks")

with Session() as s:
    n = upsert_chunks(s, all_records)
    s.commit()
print("upserted:", n)


In [ ]:
with Session() as s:
    rows = s.execute(
        select(ChunkRow.chunk_id, ChunkRow.source, ChunkRow.rule,
               ChunkRow.parent_path, ChunkRow.display_text)
        .where(ChunkRow.chunk_id.like("nbtest:real:%")).limit(4)
    ).all()
for r in rows:
    print(f"[{r.source}] {r.rule:22} {r.chunk_id}")
    print("parent_path:", r.parent_path[:70])
    print("display_text:", r.display_text[:70].replace(chr(10), " / "))


Cleanup

In [ ]:
with Session() as s:
    deleted = s.execute(
        ChunkRow.__table__.delete().where(ChunkRow.chunk_id.like("nbtest:%"))
    ).rowcount
    s.commit()
print("deleted:", deleted)

with Session() as s:
    remaining = s.execute(select(func.count()).select_from(ChunkRow)).scalar_one()
print("rows left in chunk table:", remaining)
